# Nearest-Neighbor Consistency Check

Tests whether the SimCLR embedding geometry reflects ground-truth labels — without those labels ever being used during training.

**Method:** For a random sample of players, find their *k* nearest neighbors in the 256-dim embedding space (cosine distance). Measure how often a neighbor shares the same label (role, hero, lane outcome, etc.) compared to what you'd expect by random chance.

**Key metric — enrichment:** `recall@k / chance`
- `1.0` = no better than random
- `3.0×` = neighbors share your label 3× more often than chance

The model was trained with **no labels**. Any enrichment above 1.0 is structure the encoder discovered on its own.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Works whether the notebook is opened from project root or evaluation/
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'evaluation' else cwd
sys.path.insert(0, str(project_root / 'training'))
sys.path.insert(0, str(project_root / 'evaluation'))

from umap_embeddings import load_aligned, load_model, build_embeddings
from nn_consistency import check_consistency

## Configuration

In [ ]:
DATA_PATH       = project_root / 'data' / 'matches.db'
CHECKPOINT_PATH = project_root / 'checkpoints' / 'checkpoint_best.pt'
FIGURES_DIR     = project_root / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

K          = 25     # nearest neighbors per query
N_SAMPLE   = 5000   # query points (sampled randomly)
BATCH_SIZE = 512
SEED       = 42

LABELS = ['position', 'laneOutcome', 'heroName']

## Load model and encode all players

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model, ckpt = load_model(CHECKPOINT_PATH, device)
ts_t, sc_t, labels_df = load_aligned(DATA_PATH, max_players=None, ckpt=ckpt)
print(f'Players loaded: {len(labels_df):,}')

embeddings = build_embeddings(model, ts_t, sc_t, BATCH_SIZE, device)
print(f'Embedding matrix: {embeddings.shape}')

## Run consistency check

In [ ]:
rng = np.random.default_rng(SEED)
results = check_consistency(embeddings, labels_df, LABELS, k=K, n_sample=N_SAMPLE, rng=rng)

## Results

### Role (position)

In [ ]:
ROLE_LABELS = {
    'POSITION_1': 'Pos 1 — Safe carry',
    'POSITION_2': 'Pos 2 — Mid',
    'POSITION_3': 'Pos 3 — Offlaner',
    'POSITION_4': 'Pos 4 — Soft support',
    'POSITION_5': 'Pos 5 — Hard support',
}

df_pos = results['position'].copy()
df_pos['role'] = df_pos['class'].map(ROLE_LABELS).fillna(df_pos['class'])
df_pos_display = df_pos[['role', 'recall_at_k', 'chance', 'enrichment', 'n_queries']].copy()

mean_enrich = df_pos['enrichment'].mean()
print(f'Mean enrichment (position): {mean_enrich:.2f}×  (chance baseline: 1.00×)\n')
df_pos_display.style \
    .format({'recall_at_k': '{:.3f}', 'chance': '{:.3f}', 'enrichment': '{:.2f}×'}) \
    .bar(subset='enrichment', color='#5b9bd5', vmin=0) \
    .hide(axis='index')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ['#e15759' if e == df_pos['enrichment'].max() else '#5b9bd5' for e in df_pos['enrichment']]
bars = ax.barh(df_pos['role'], df_pos['enrichment'], color=colors)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Random chance')
ax.bar_label(bars, fmt='{:.2f}×', padding=4, fontsize=9)
ax.set_xlabel(f'Enrichment (recall@{K} / chance)')
ax.set_title(f'Role clustering in SimCLR embedding space  (k={K}, n={N_SAMPLE:,})')
ax.legend(fontsize=8)
ax.set_xlim(0, df_pos['enrichment'].max() * 1.18)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nn_position_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()

### Lane outcome

In [ ]:
df_lane = results['laneOutcome']
mean_enrich = df_lane['enrichment'].mean()
print(f'Mean enrichment (laneOutcome): {mean_enrich:.2f}×\n')
df_lane.style \
    .format({'recall_at_k': '{:.3f}', 'chance': '{:.3f}', 'enrichment': '{:.2f}×'}) \
    .bar(subset='enrichment', color='#5b9bd5', vmin=0) \
    .hide(axis='index')

### Hero (top 20 by enrichment)

In [ ]:
df_hero = results['heroName'].head(20)
mean_enrich = results['heroName']['enrichment'].mean()
print(f'Mean enrichment across all heroes: {mean_enrich:.2f}×  (chance baseline ≈ {results["heroName"]["chance"].mean():.3f})\n')
df_hero.style \
    .format({'recall_at_k': '{:.3f}', 'chance': '{:.3f}', 'enrichment': '{:.2f}×'}) \
    .bar(subset='enrichment', color='#70ad47', vmin=0) \
    .hide(axis='index')

In [ ]:
df_hero_all = results['heroName']
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_hero_all['enrichment'], bins=30, color='#70ad47', edgecolor='white', linewidth=0.5)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Random chance')
ax.axvline(df_hero_all['enrichment'].mean(), color='#e15759', linestyle='-', linewidth=1.5,
           label=f'Mean ({df_hero_all["enrichment"].mean():.2f}×)')
ax.set_xlabel(f'Enrichment (recall@{K} / chance)')
ax.set_ylabel('Number of heroes')
ax.set_title('Per-hero enrichment distribution')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nn_hero_enrichment_hist.png', dpi=150, bbox_inches='tight')
plt.show()